In [2]:
import pandas as pd
import numpy as np
from natasha import NamesExtractor, MorphVocab
import pymorphy3
import re

In [3]:
df = pd.read_excel('export.xlsx')
df

/Users/sergej/PyCharmMiscProject/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Реестр иностранных агентов,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,2025-09-05 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,№ п/п,Полное наименование (прежнее наименование (в с...,Основания для включения,Дата принятия Минюстом России решения о включе...,Дата принятия Минюстом России решения об исклю...,Доменное имя информационного ресурса (при нали...,Тип иностранного агента,Регистрационный номер,ИНН,ОГРН,...,Дата рождения,Полное наименование или ФИО участников,Адрес (место нахождения),Дата опубликования принятого Минюстом России р...,Номер специального счета,Наименование и местонахождение уполномоченного...,Банковский идентификационный код уполномоченно...,Номер корреспондентского счета (субсчета) упол...,Дата открытия специального счета,Дата заключения договора банковского счёта
2,1068,Сетевой проект «Om TV»,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://www.youtube.com/@omtvreal; https://t.m...,Иные объединения лиц,NaN,NaN,NaN,...,NaN,[Омельчук Сергей Сергеевич],NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
3,1067,«Компромат 1»,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://kompromat1.online/; https://kompromat1...,Иные объединения лиц,NaN,NaN,NaN,...,NaN,"[Черненко Константин Евгеньевич, Преснаков Але...",NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
4,1066,Степанова Анна Васильевна,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://t.me/markizaakarabasa; https://www.you...,Физические лица,NaN,292006666153,NaN,...,28.10.1979,NaN,NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1065,5,Региональная общественная организация в защиту...,Статья 32 Федерального закона от 12.01.1996 № ...,05.06.2014,20.02.2017,NaN,Юридические лица,NaN,7709439312,1067799008860,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1066,4,Автономная некоммерческая научно-исследователь...,Статья 32 Федерального закона от 12.01.1996 № ...,05.06.2014,22.05.2015,NaN,Юридические лица,NaN,6452083317,1036405207640,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1067,3,Региональная общественная правозащитная органи...,Статья 32 Федерального закона от 12.01.1996 № ...,05.06.2014,29.02.2016,NaN,Юридические лица,NaN,6150025245,1026100023872,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1068,2,Ассоциация некоммерческих организаций «В защит...,Статья 32 Федерального закона от 12.01.1996 № ...,05.06.2014,13.03.2020,NaN,Юридические лица,NaN,7702295527,1037739618872,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
#оставляем только имена - удаляем организации. Принцип - "Тип иностранного агента" - физ лицо

df = df[df['Unnamed: 6'] == "Физические лица"]
df

,Реестр иностранных агентов,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
4,1066,Степанова Анна Васильевна,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://t.me/markizaakarabasa; https://www.you...,Физические лица,NaN,292006666153,NaN,...,28.10.1979,NaN,NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
5,1065,Смолин Владимир Александрович,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://t.me/smolin_info; https://www.facebook...,Физические лица,NaN,240701390600,NaN,...,14.10.1981,NaN,NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
6,1064,Ицхоки Олег Евгеньевич,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://itskhoki.com; https://www.instagram.co...,Физические лица,NaN,771402711994,NaN,...,07.01.1983,NaN,NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
7,1063,Говорун Сергей Николаевич,Статья 7 Федерального закона от 14.07.2022 № 2...,05.09.2025,NaN,https://t.me/cyrilhovorun; https://www.faceboo...,Физические лица,NaN,NaN,NaN,...,28.01.1974,NaN,NaN,05.09.2025,NaN,NaN,NaN,NaN,NaN,NaN
8,1062,"Несмиян Анатолий Евгеньевич ""El Murid""",Статья 7 Федерального закона от 14.07.2022 № 2...,29.08.2025,NaN,"https://t.me/anatoly_nesmiyan, ID: -1001540992...",Физические лица,NaN,165002994679,NaN,...,08.11.1965,NaN,NaN,29.08.2025,40817810255193220413,Головное отделение Северо-Западного банка №905...,044030653,30101810500000000653,03.09.2025,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
854,216,Савицкая Людмила Алексеевна,Статья 6 Закона Российской Федерации от 27.12....,28.12.2020,NaN,https://vk.com/id25797959; https://lsavitskaya...,Физические лица,NaN,602722507030,NaN,...,22.01.1991,NaN,NaN,NaN,40817810751861520329,Псковское отделение №8630 ПАО Сбербанк,045805602,30101810300000000602,01.03.2025,NaN
855,215,Пономарев Лев Александрович,Статья 6 Закона Российской Федерации от 27.12....,28.12.2020,NaN,https://www.youtube.com/@levzaprava; https://t...,Физические лица,NaN,773404771583,NaN,...,02.09.1941,NaN,NaN,NaN,40817810638700402181,Московский банк ПАО Сбербанк,044525225,30101810400000000225,01.03.2025,NaN
856,214,Маркелов Сергей Евгеньевич,Статья 6 Закона Российской Федерации от 27.12....,28.12.2020,NaN,http://xamin.tilda.ws/cv?fbclid=IwAR0VahYgErIP...,Физические лица,NaN,100122365435,NaN,...,11.10.1986,NaN,NaN,NaN,40817810225861797882,Карельское отделение №8628 ПАО Сбербанк,048602673,30101810600000000673,01.03.2025,NaN
857,213,Камалягин Денис Николаевич,Статья 6 Закона Российской Федерации от 27.12....,28.12.2020,NaN,https://t.me/guberniaband; https://t.me/iwantm...,Физические лица,NaN,602723196108,NaN,...,22.02.1985,NaN,NaN,NaN,40817810451861520328,Псковское отделение №8630 ПАО Сбербанк,045805602,30101810300000000602,01.03.2025,NaN


In [5]:
test_cases = [
    'Иванов Иван Иванович',
    'Иванов Иван Иванович "ivanov"',
    'Иванов Иван "ivan"',
    'Петров Петr',
    '"superuser" Сидоров Сидор Сидорович',
    '"only_nickname"',
    'Смирнова Анна "анютка"',
    'Кузнецов "kuznec"',
    'Иванов Иван Иванович "Петя Рыбкин"',  # новый тест-кейс
    'Константинов Костя Константинович "El\'Loh"'  # новый тест-кейс
]

def is_informative_nickname(nickname):
    """
    Проверяет, является ли никнейм информативным (содержит кириллицу и пробелы)
    или неинформативным (латиница, цифры, специальные символы)
    """
    # Если содержит кириллические символы и пробелы - информативный
    if re.search(r'[а-яёА-ЯЁ]', nickname) and ' ' in nickname:
        return True
    
    # Если содержит только латиницу, цифры и специальные символы - неинформативный
    if re.match(r'^[a-zA-Z0-9_\-\'\@\.]+$', nickname) and not re.search(r'[а-яёА-ЯЁ]', nickname):
        return False
    
    # Во всех остальных случаях считаем информативным
    return True

def universal_extract_names(full_name):
    """
    Универсальная функция для извлечения всех имен, включая информативные никнеймы
    """
    if not full_name or not isinstance(full_name, str):
        return []
    
    # Извлекаем основной текст и никнеймы в кавычках
    main_part = re.sub(r'"[^"]*"', '', full_name).strip()
    nicknames = re.findall(r'"([^"]*)"', full_name)
    
    result = []
    
    # Добавляем основное имя (без кавычек)
    if main_part:
        result.append(main_part)
    
    # Добавляем информативные никнеймы
    for nickname in nicknames:
        if is_informative_nickname(nickname):
            result.append(nickname)
    
    return result

# Тестирование
print("Универсальная версия:")
for test in test_cases:
    result = universal_extract_names(test)
    print(f"'{test}' -> {result}")

# Дополнительные тесты
additional_tests = [
    'Иванов Иван "Петр Васильев"',  # информативный ник
    'Сидоров "john_doe123"',  # неинформативный ник
    '"Анна Каренина" Петрова',  # информативный ник в начале
    'Кузнецов "мастер88"',  # смешанный - не информативный
]

print("\nДополнительные тесты:")
for test in additional_tests:
    result = universal_extract_names(test)
    print(f"'{test}' -> {result}")

Универсальная версия:
'Иванов Иван Иванович' -> ['Иванов Иван Иванович']
'Иванов Иван Иванович "ivanov"' -> ['Иванов Иван Иванович']
'Иванов Иван "ivan"' -> ['Иванов Иван']
'Петров Петr' -> ['Петров Петr']
'"superuser" Сидоров Сидор Сидорович' -> ['Сидоров Сидор Сидорович']
'"only_nickname"' -> []
'Смирнова Анна "анютка"' -> ['Смирнова Анна', 'анютка']
'Кузнецов "kuznec"' -> ['Кузнецов']
'Иванов Иван Иванович "Петя Рыбкин"' -> ['Иванов Иван Иванович', 'Петя Рыбкин']
'Константинов Костя Константинович "El'Loh"' -> ['Константинов Костя Константинович']

Дополнительные тесты:
'Иванов Иван "Петр Васильев"' -> ['Иванов Иван', 'Петр Васильев']
'Сидоров "john_doe123"' -> ['Сидоров']
'"Анна Каренина" Петрова' -> ['Петрова', 'Анна Каренина']
'Кузнецов "мастер88"' -> ['Кузнецов', 'мастер88']


In [6]:
def create_last_name_variations(last_names):
    """
    Создает возможные варианты склонения фамилий
    """
    morph = pymorphy3.MorphAnalyzer()
    all_variations = {}
    
    for last_name in last_names:
        # Парсим фамилию
        parsed = morph.parse(last_name)[0]
        
        # Добавляем нормальную форму
        variations = {parsed.normal_form.lower(): last_name}
        
        # Генерируем варианты в разных падежах
        cases = ['nomn', 'gent', 'datv', 'accs', 'ablt', 'loct']
        
        for case in cases:
            try:
                inflected = parsed.inflect({case})
                if inflected:
                    variations[inflected.word.lower()] = last_name
            except:
                continue
        
        # Объединяем варианты
        for variation, original in variations.items():
            all_variations[variation] = original
    
    return all_variations

def find_last_name_in_text(text, target_last_names):
    """
    Ищет фамилии в тексте и возвращает список найденных фамилий
    """
    if not text or not target_last_names:
        return []
    
    # Создаем множество всех возможных вариантов фамилий
    all_variations = create_last_name_variations(target_last_names)
    
    # Приводим текст к нижнему регистру и ищем совпадения
    text_lower = text.lower()
    words = re.findall(r'\b[а-яё]+\b', text_lower)
    
    found_last_names = set()
    
    for word in words:
        if word in all_variations:
            found_last_names.add(all_variations[word])
    
    return list(found_last_names)

# Пример использования
text = "Вчера я видел Иванова, который разговаривал с Ивановой о делах Петрова"
targets = ["Иванов", "Петров", "Сидоров"]

result = find_last_name_in_text(text, targets)
print(f"Найдены фамилии: {result}")

# Покажем сгенерированные варианты
print("\nВарианты фамилий:")
variations = create_last_name_variations(["Иванов", "Петров", "Моргенштерн"])
for variation, original in variations.items():
    print(f"{variation} -> {original}")

Найдены фамилии: ['Петров', 'Иванов']

Варианты фамилий:
иванов -> Иванов
иванова -> Иванов
иванову -> Иванов
ивановым -> Иванов
иванове -> Иванов
петров -> Петров
петрова -> Петров
петрову -> Петров
петровым -> Петров
петрове -> Петров
моргенштерн -> Моргенштерн
моргенштерна -> Моргенштерн
моргенштерну -> Моргенштерн
моргенштерном -> Моргенштерн
моргенштерне -> Моргенштерн


In [7]:
stop_list = df['Unnamed: 1'].to_list()
print("Исходный stop_list:")
print(stop_list)

morph_vocab = MorphVocab()
extractor = NamesExtractor(morph_vocab)

def extract_and_reorder_surname(full_name):
    """
    Извлекает фамилию и при необходимости переставляет её на первое место
    """
    # Извлекаем фамилию с помощью Natasha
    matches = list(extractor(full_name))
    if not matches:
        return full_name  # Если не нашли фамилию, возвращаем как есть
    
    match = matches[0].fact
    surname = match.last
    
    if not surname:
        return full_name
    
    # Разбиваем полное имя на слова
    words = full_name.split()
    
    # Если фамилия уже на первом месте, возвращаем как есть
    if words[0] == surname:
        return full_name
    
    # Если фамилия найдена, но не на первом месте - переставляем
    if surname in words:
        # Удаляем фамилию из текущей позиции
        words.remove(surname)
        # Вставляем фамилию на первое место
        words.insert(0, surname)
        return ' '.join(words)
    
    return full_name

def extract_surname(full_name):
    """
    Извлекает только фамилию
    """
    matches = list(extractor(full_name))
    if matches:
        match = matches[0].fact
        return match.last
    return None

# Получаем ВСЕ имена и информативные псевдонимы
all_names_from_stoplist = []

for name in stop_list:
    names_list = universal_extract_names(name)
    all_names_from_stoplist.extend(names_list)

print("\nВсе имена и информативные псевдонимы:")
print(all_names_from_stoplist)

# Обрабатываем каждое имя: переставляем фамилию на первое место если нужно
reordered_names = []
for full_name in all_names_from_stoplist:
    reordered_name = extract_and_reorder_surname(full_name)
    reordered_names.append(reordered_name)

print("\nИмена с фамилиями на первом месте:")
print(reordered_names)

# Извлекаем только фамилии
stop_list_lastnames = []
for full_name in reordered_names:
    surname = extract_surname(full_name)
    if surname:
        stop_list_lastnames.append(surname)

print("\nТолько фамилии из всех имен и псевдонимов:")
print(stop_list_lastnames)

Исходный stop_list:
['Степанова Анна Васильевна', 'Смолин Владимир Александрович', 'Ицхоки Олег Евгеньевич', 'Говорун Сергей Николаевич', 'Несмиян Анатолий Евгеньевич "El Murid"', 'Кузнецов Борис Аврамович', 'Кротенко Евгений Андреевич', 'Кривцова Олеся Романовна', 'Везикко Ирия Валтеровна', 'Бобров Юрий Сергеевич', 'Нюберг Дмитрий Сергеевич "Qianti"', 'Марков Сергей Александрович', 'Кордочкин Андрей Борисович', 'Дугарова Раджана Дашинимаевна', 'Солонин Марк Семенович', 'Рудников Игорь Петрович', 'Курмояров Иоанн Валерьевич', 'Кротов Марк Яковлевич "Крутов"', 'Сивенок Андрей Александрович', 'Мысина Оксана Анатольевна', 'Мовчан Андрей Андреевич', 'Долиев Михаил Вячеславович', 'Храмцов Дмитрий Александрович', 'Торстрем Ксения Владимировна', 'Резник Наталья Рэмовна', 'Ольшевец Фелицата Дмитриевна "Маша Майерс"', 'Кириллова Ксения Валерьевна', 'Жарков Василий Павлович', 'Великий Дмитрий Сергеевич', 'Росов Николай Вячеславович', 'Поташов Валерий Николаевич', 'Литвишко Дарья Алексеевна', 'Зу

In [8]:
print([all_names_from_stoplist[i] for i in range(len(stop_list_lastnames)) if stop_list_lastnames[i] is None])

[]


In [9]:
stop_list_lastnames = [name for name in stop_list_lastnames if name is not None]

In [11]:
with open("inoagents_050925.txt", "w", encoding="utf-8") as file:
    file.writelines(', '.join(stop_list_lastnames))

with open("ia_initials_050925.txt", "w", encoding="utf-8") as file:
    file.writelines(', '.join(reordered_names))

print("Список сохранен в файл my_list.txt")

Список сохранен в файл my_list.txt
